# 07 — Post-hoc Test Comparison (descriptive only)

**Vietnamese Punctuation Restoration · Phase 2, notebook 3 / 3**

> ## The final winner was selected exclusively using validation results before this notebook was executed.
>
> ## The following official-test comparison is descriptive post-hoc analysis only and must not be used to change the selected winner.

---

## 1. Notebook này để làm gì, và không để làm gì

**Để làm gì.** Sau khi winner đã chốt và official test đã chạy, việc chấm nốt
các model còn lại trên test cho ta bức tranh đầy đủ: chênh lệch giữa bốn model
trên test có giống trên validation không? Class weight tác động thế nào? Model
học được có hơn baseline thủ công không? Đây là thông tin có giá trị cho phần
thảo luận của báo cáo.

**Không để làm gì.** Không để chọn lại winner. Nếu bảng dưới cho thấy một model
khác cao điểm hơn winner trên test, winner **vẫn giữ nguyên**. Lý do:

* điểm test chỉ là ước lượng không thiên lệch **khi nó không tham gia vào quyết
  định nào**. Chọn model theo test rồi báo cáo chính điểm test đó là tự lừa
  mình;
* chênh lệch nhỏ giữa các model chủ yếu là nhiễu. Chọn cái cao nhất trên một
  lần đo chính là "overfit lên test set".

Cell đầu tiên sẽ **kiểm tra và dừng lại** nếu winner chưa được khoá.

## 2. Có gì trong bảng so sánh

| | |
|---|---|
| `B0` | baseline đa số — đoán `O` cho mọi từ |
| `B1` | baseline heuristic từ khoá tiếng Việt (không dùng dữ liệu huấn luyện) |
| `E1` | BiLSTM from scratch |
| `E2` | PhoBERT, không class weight |
| `E3` | PhoBERT, inverse weight |
| `E4` | PhoBERT, sqrt-inverse weight |

Hai baseline cho biết "sàn" của bài toán: nếu một model không vượt xa `B1` thì
việc huấn luyện nó không mang lại gì.

In [1]:
import os, sys, json, time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), f"Cannot locate the repo root from {Path.cwd()}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from src.data.constants import LABELS, PUNCTUATION_LABELS, EXPERIMENT_IDS, OUTPUTS_DIR
from src.utils.io import read_json, write_json, write_csv
from src.utils.logging_utils import configure_stdout_utf8

configure_stdout_utf8()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EVALUATION_DIR = OUTPUTS_DIR / "evaluation"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: .


In [2]:

from src.evaluation.selection import load_locked_winner

selection = load_locked_winner()
assert selection["winner_locked"] is True, "Winner chưa bị khoá — không được chạy post-hoc."
assert selection["test_was_used_for_selection"] is False

WINNER = selection["winner"]
final_test_path = EVALUATION_DIR / "final_test_results.json"
assert final_test_path.exists(), (
    "Chưa chạy notebook 06. Official test của winner phải chạy trước post-hoc."
)

print("Winner đã khoá  :", WINNER)
print("Khoá lúc (UTC)  :", selection["locked_at_utc"])
print("Chọn bằng       :", selection["selection_split"], "/", selection["selection_metric"])
print("Test dùng để chọn:", selection["test_was_used_for_selection"])
print("\nĐược phép chạy post-hoc comparison.")

Winner đã khoá  : E2
Khoá lúc (UTC)  : 2026-08-09T22:59:40+00:00
Chọn bằng       : validation / punctuation_macro_f1
Test dùng để chọn: False

Được phép chạy post-hoc comparison.


## 3. Chấm baseline trên test

`B0` và `B1` không cần GPU và không dùng dữ liệu huấn luyện, nên chấm rất
nhanh. Luật của `B1` được in ra đầy đủ để báo cáo trích dẫn được.

In [3]:
from src.data.constants import PROCESSED_FILES
from src.data.dataset import load_examples
from src.evaluation.baselines import build_baseline, evaluate_baseline, B1_RULES_DESCRIPTION

test_examples = load_examples(PROCESSED_FILES["test"])
print(f"Test: {len(test_examples):,} examples, "
      f"{sum(len(e.tokens) for e in test_examples):,} words\n")

print(B1_RULES_DESCRIPTION)

baseline_results = {}
for bid in ("B0", "B1"):
    b = build_baseline(bid)
    started = time.time()
    m = evaluate_baseline(b, test_examples)
    baseline_results[bid] = m
    print(f"\n{bid} — {b.name}  ({time.time() - started:.1f}s)")
    print(f"   accuracy {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | "
          f"PUNCT-F1 {m['punctuation_macro_f1']:.4f}")

2026-08-10 06:02:01 | INFO    | src.data.dataset | Loaded 22832 examples from test.jsonl


Test: 22,832 examples, 2,968,815 words

B1 — Vietnamese cue-word heuristic (deterministic, no training data)

Scanning left to right and tracking `n` = number of words since the last
predicted PERIOD/QUESTION, the label of word `w[i]` is the first rule that
matches:

  1. QUESTION  if w[i] ∈ QUESTION_FINAL_CUES and n >= 4
                 and (i is the last word, or w[i+1] ∉ QUESTION_FINAL_CUES)
  2. PERIOD    if w[i] ∈ SENTENCE_END_CUES and n >= 6
  3. PERIOD    if n >= 25                (run-on guard)
  4. COMMA     if w[i+1] ∈ CLAUSE_START_CUES and n >= 4
  5. O         otherwise

The final word of a chunk is forced to PERIOD if it would otherwise be O,
because a chunk boundary is (almost always) a sentence boundary in this data.

Cue lists were written from Vietnamese grammar, not tuned on any split.



B0 — Majority class (all O)  (0.4s)
   accuracy 0.9104 | macro-F1 0.2383 | PUNCT-F1 0.0000



B1 — Vietnamese cue-word heuristic  (1.2s)
   accuracy 0.8312 | macro-F1 0.3526 | PUNCT-F1 0.1660


## 4. Chấm bốn model đã huấn luyện trên test

Mỗi checkpoint được nạp từ đĩa, chấm trên toàn bộ test, rồi giải phóng khỏi GPU
trước khi nạp model tiếp theo — GPU 8GB không chứa được hai PhoBERT cùng lúc.

Với winner, kết quả ở đây phải **trùng khớp** với `final_test_results.json` của
notebook 06; cell cuối kiểm tra điều đó.

In [4]:
import torch
from src.evaluation.evaluator import evaluate
from src.evaluation.loaders import build_eval_dataloader
from src.models.factory import load_model_from_checkpoint
from src.data.constants import CHECKPOINT_DIR, EXPERIMENT_IDS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
print("Device:", device, "\n")

model_results = {}
for exp in EXPERIMENT_IDS:
    ckpt = CHECKPOINT_DIR / exp
    started = time.time()
    model, meta = load_model_from_checkpoint(ckpt, device=device)
    loader, _, _ = build_eval_dataloader(ckpt, test_examples, batch_size=32)
    m = evaluate(model, loader, device=device, loss_fn=criterion,
                 use_amp=(device.type == "cuda"), desc=f"{exp} test")
    model_results[exp] = m
    print(f"{exp}: PUNCT-F1 {m['punctuation_macro_f1']:.6f} | "
          f"acc {m['accuracy']:.6f} | macro-F1 {m['macro_f1']:.6f} "
          f"({time.time() - started:.0f}s)")
    del model, loader
    if device.type == "cuda":
        torch.cuda.empty_cache()

Device: cuda 

2026-08-10 06:02:03 | INFO    | src.evaluation.loaders | Loaded BiLSTM vocabulary (31615 types) from E1


E1 test:   0%|          | 0/714 [00:00<?, ?it/s]

E1: PUNCT-F1 0.624629 | acc 0.944153 | macro-F1 0.711906 (6s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-10 06:02:12 | INFO    | src.evaluation.loaders | Loaded PhoBERT tokenizer from E2 (max_length=192)


...

2026-08-10 06:03:34 | INFO    | src.evaluation.loaders | Loaded PhoBERT tokenizer from E4 (max_length=192)


2026-08-10 06:03:36 | INFO    | src.data.dataset | Encoded 20000/22832 examples


2026-08-10 06:03:36 | INFO    | src.data.dataset | PhoBERT dataset ready: {'num_examples': 22832, 'num_windows': 22857, 'num_words': 2968815, 'num_subwords': 3129144, 'windows_per_example': 1.0011, 'subwords_per_word': 1.054, 'examples_split_into_multiple_windows': 25, 'max_window_length': 192}


E4 test:   0%|          | 0/715 [00:00<?, ?it/s]

E4: PUNCT-F1 0.743845 | acc 0.956761 | macro-F1 0.803218 (42s)


## 5. Bảng post-hoc đầy đủ

Cột `validation_punct_f1` được đưa vào để so hai bảng xếp hạng. Nếu thứ tự trên
validation và trên test giống nhau, đó là bằng chứng tốt cho thấy việc chọn
model bằng validation là đáng tin. Nếu khác nhau, đó là điều **cần thảo luận
trong báo cáo** — chứ không phải lý do để đổi winner.

In [5]:
val_rows = {r["experiment_id"]: r for r in
            read_json(EVALUATION_DIR / "validation_model_comparison.json")["rows"]}

rows = []
for bid, m in baseline_results.items():
    rows.append({
        "system": bid, "type": "baseline",
        "weight_mode": "-", "validation_punct_f1": None,
        "test_punct_f1": round(m["punctuation_macro_f1"], 6),
        "test_accuracy": round(m["accuracy"], 6),
        "test_macro_f1": round(m["macro_f1"], 6),
        **{f"test_f1_{l.lower()}": round(m["per_class"][l]["f1"], 6) for l in LABELS},
    })
for exp, m in model_results.items():
    v = val_rows[exp]
    rows.append({
        "system": exp, "type": "trained model",
        "weight_mode": v["weight_mode"],
        "validation_punct_f1": round(v["validation_punctuation_macro_f1"], 6),
        "test_punct_f1": round(m["punctuation_macro_f1"], 6),
        "test_accuracy": round(m["accuracy"], 6),
        "test_macro_f1": round(m["macro_f1"], 6),
        **{f"test_f1_{l.lower()}": round(m["per_class"][l]["f1"], 6) for l in LABELS},
    })

posthoc = pd.DataFrame(rows).sort_values("test_punct_f1", ascending=False).reset_index(drop=True)
posthoc.insert(0, "test_rank", range(1, len(posthoc) + 1))
posthoc["is_locked_winner"] = posthoc["system"] == WINNER
display(posthoc)

write_csv(EVALUATION_DIR / "posthoc_test_model_comparison.csv",
          list(posthoc.columns), posthoc.values.tolist())
write_json(EVALUATION_DIR / "posthoc_test_model_comparison.json", {
    "disclaimer": ("The final winner was selected exclusively using validation results "
                   "before this comparison was executed. This table is descriptive "
                   "post-hoc analysis only and did not change the selected winner."),
    "locked_winner": WINNER,
    "rows": posthoc.to_dict(orient="records"),
})
print("Written: outputs/evaluation/posthoc_test_model_comparison.{csv,json}")

,test_rank,system,type,weight_mode,validation_punct_f1,test_punct_f1,test_accuracy,test_macro_f1,test_f1_o,test_f1_comma,test_f1_period,test_f1_question,is_locked_winner
0,1,E2,trained model,none,0.778718,0.776322,0.966265,0.828767,0.986099,0.722621,0.809131,0.797215,True
1,2,E4,trained model,sqrt_inverse,0.745676,0.743845,0.956761,0.803218,0.981339,0.697223,0.788731,0.745580,False
2,3,E3,trained model,inverse,0.695324,0.694776,0.935458,0.763427,0.969381,0.610662,0.756901,0.716764,False
3,4,E1,trained model,none,0.628241,0.624629,0.944153,0.711906,0.973737,0.489322,0.683727,0.700838,False
4,5,B1,baseline,-,NaN,0.166006,0.831232,0.352572,0.912270,0.086179,0.198948,0.212890,False
5,6,B0,baseline,-,NaN,0.000000,0.910424,0.238278,0.953112,0.000000,0.000000,0.000000,False


Written: outputs/evaluation/posthoc_test_model_comparison.{csv,json}


In [6]:

trained = posthoc[posthoc["type"] == "trained model"].sort_values("system")
base    = posthoc[posthoc["type"] == "baseline"].sort_values("system")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(trained))
ax.bar(x - 0.2, trained["validation_punct_f1"], 0.4, label="validation", color="#9ecae1")
ax.bar(x + 0.2, trained["test_punct_f1"], 0.4, label="official test", color="#2a7fb8")
for xi, v in zip(x - 0.2, trained["validation_punct_f1"]):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
for xi, v in zip(x + 0.2, trained["test_punct_f1"]):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
for i, s in enumerate(trained["system"]):
    if s == WINNER:
        ax.annotate("WINNER\n(chọn bằng validation)", (i, 0.06), ha="center",
                    fontsize=8, color="#1a7a3c", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(trained["system"])
ax.set_ylabel("Punctuation Macro-F1"); ax.set_ylim(0, 1.0)
ax.set_title("Validation vs official test — 4 model đã huấn luyện")
ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=9)

ax = axes[1]
allsys = posthoc.sort_values("test_punct_f1")
colors = ["#1a7a3c" if s == WINNER else ("#bbbbbb" if t == "baseline" else "#2a7fb8")
          for s, t in zip(allsys["system"], allsys["type"])]
ax.barh(allsys["system"], allsys["test_punct_f1"], color=colors)
for i, v in enumerate(allsys["test_punct_f1"]):
    ax.text(v + 0.008, i, f"{v:.4f}", va="center", fontsize=9)
ax.set_xlabel("Punctuation Macro-F1 (official test)")
ax.set_xlim(0, 1.0)
ax.set_title("Tất cả hệ thống trên test (xanh lá = winner đã khoá)")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "posthoc_test_model_comparison.png", dpi=150)
plt.show()
print("Saved:", (FIGURES_DIR / "posthoc_test_model_comparison.png").relative_to(PROJECT_ROOT))

Saved: outputs\figures\posthoc_test_model_comparison.png


<USER_HOME>/AppData\Local\Temp\ipykernel_21736\850554684.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Winner có bị đổi không?

Không. Cell dưới nêu rõ thứ hạng của winner trên test và khẳng định lại rằng
`model_selection.json` không hề bị chỉnh sửa.

Nếu winner **không** đứng đầu bảng test, đó là một quan sát thú vị đáng viết
vào báo cáo (thường là do chênh lệch nhỏ trong khoảng nhiễu, hoặc do phân bố
test hơi khác validation) — nhưng nó không làm thay đổi quyết định đã chốt.

In [7]:
winner_row = posthoc[posthoc["system"] == WINNER].iloc[0]
best_row   = posthoc.iloc[0]

print(f"Winner đã khoá         : {WINNER}")
print(f"  hạng trên test        : {int(winner_row['test_rank'])} / {len(posthoc)}")
print(f"  test PUNCT-F1         : {winner_row['test_punct_f1']:.6f}")
print(f"  validation PUNCT-F1   : {winner_row['validation_punct_f1']:.6f}")
print()
print(f"Hệ thống test cao nhất : {best_row['system']} ({best_row['test_punct_f1']:.6f})")

if best_row["system"] != WINNER:
    print("\n  LƯU Ý: một hệ thống khác đạt điểm test cao hơn winner.")
    print(f"  Chênh lệch: {best_row['test_punct_f1'] - winner_row['test_punct_f1']:+.6f}")
    print("  Winner KHÔNG thay đổi. Điểm test không được dùng để chọn model;")
    print("  hiện tượng này chỉ được ghi nhận và thảo luận trong báo cáo.")
else:
    print("\n  Winner cũng dẫn đầu trên test — validation và test đồng thuận.")


after = read_json(EVALUATION_DIR / "model_selection.json")
assert after["winner"] == WINNER and after["winner_locked"] is True
assert after["test_was_used_for_selection"] is False
print(f"\n  model_selection.json vẫn nguyên vẹn: winner={after['winner']}, "
      f"locked={after['winner_locked']}, test_used={after['test_was_used_for_selection']}")

Winner đã khoá         : E2
  hạng trên test        : 1 / 6
  test PUNCT-F1         : 0.776322
  validation PUNCT-F1   : 0.778718

Hệ thống test cao nhất : E2 (0.776322)

  Winner cũng dẫn đầu trên test — validation và test đồng thuận.

  model_selection.json vẫn nguyên vẹn: winner=E2, locked=True, test_used=False


In [8]:

final06 = read_json(EVALUATION_DIR / "final_test_results.json")
here = model_results[WINNER]["punctuation_macro_f1"]
there = final06["metrics"]["punctuation_macro_f1"]
print(f"Winner test PUNCT-F1  — notebook 06: {there:.6f}")
print(f"                       notebook 07: {here:.6f}")
print(f"  chênh lệch: {abs(here - there):.2e}  "
      f"({'khớp' if abs(here - there) < 1e-3 else 'KHÔNG KHỚP — cần kiểm tra'})")

print("\n" + "=" * 78)
print("POST-HOC COMPARISON COMPLETE — descriptive only, winner unchanged.")
print("=" * 78)

Winner test PUNCT-F1  — notebook 06: 0.776322
                       notebook 07: 0.776322
  chênh lệch: 0.00e+00  (khớp)

POST-HOC COMPARISON COMPLETE — descriptive only, winner unchanged.
